<a href="https://colab.research.google.com/github/takahashi-crypto556/gemini-app/blob/main/04_business_card_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [69]:
!pip install google-genai

In [70]:
import os
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='こんにちは！開発環境の接続テストです。応答できますか？',
)

print(response.text)


こんにちは！はい、正常に応答できています。接続テストは成功です！

開発環境の構築やコードの実装、デバッグなど、何かお手伝いできることがあればお気軽にお申し付けください。よろしくお願いします！


In [71]:
import re

def extract_value(text, field):
    key = str(field).lower()

    patterns = {
        "company_name": r"^\s*(?:company_name|会社名)[:：]\s*(.*)",
        "department":   r"^\s*(?:department|部署名|部署)[:：]\s*(.*)",
        "title":        r"^\s*(?:title|役職)[:：]\s*(.*)",
        "name":         r"^\s*(?:name|name|氏名)[:：]\s*(.*)",
        "name_kana":    r"^\s*(?:name_kana|フリガナ)[:：]\s*(.*)",
        "email":        r"^\s*(?:email|MAIL|メールアドレス)[:：]\s*(.*)",
        "phone":        r"^\s*(?:phone|TEL|電話番号)[:：]\s*(.*)",
        "address":      r"^\s*(?:address|住所)[:：]\s*(.*)",
     }

    pattern = patterns.get(key)
    if pattern:
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match and match.group(1):
            val = match.group(1).strip()
            if val:
                return val

    return None

class BusinessCard:
    def __init__(self,text):
        self.Company_name = extract_value(text, "company_name")
        self.Department = extract_value(text, "department")
        self.Title = extract_value(text, "title")
        self.Name = extract_value(text, "name")
        self.Name_kana = extract_value(text, "name_kana")
        self.Email = extract_value(text, "email")
        self.Phone = extract_value(text, "phone")
        self.Address = extract_value(text, "address")

    def __str__(self):
        lines = []
        if self.Company_name:
            lines.append("会社名:" + self.Company_name)
        if self.Department:
            lines.append("部署名:" + self.Department)
        if self.Title:
            lines.append("役職:" + self.Title)
        if self.Name:
            lines.append("氏名:" + self.Name)
        if self.Name_kana:
            lines.append("フリガナ:" + self.Name_kana)
        if self.Email:
            lines.append("メールアドレス:" + self.Email)
        if self.Phone:
            lines.append("電話番号:" + self.Phone)
        if self.Address:
            lines.append("住所:" + self.Address)
        return"\n".join(lines) if lines else "情報なし"

In [72]:
!pip install litellm

In [73]:
import base64
import os
import sys
import litellm
from google.colab import files, userdata

DEFAULT_DIR = "data"
DEFAULT_IMAGE_PATH = os.path.join(DEFAULT_DIR,"sample_card.png")

os.makedirs(DEFAULT_DIR, exist_ok=True)

print("---名刺画像の読み込み---")
print("※ファイルをアップロードするか、キャンセル/スキップして同梱のサンプル画像を使用します。")
uploaded = files.upload()

if uploaded:
    image_path = list(uploaded.keys())[0]
    print(f"アップロードされた画像を使用します:{image_path}")

    with open(DEFAULT_IMAGE_PATH, "wb") as f:
        f.write(uploaded[image_path])
else:
    if os.path.exists(DEFAULT_IMAGE_PATH):
        print(f"同梱のサンプル画像を使用します: {image_path}")
    else:
        raise FileNotFoundError(
            f"エラー: {DEFAULT_IMAGE_PATH}が見つかりません。\n"
            "初回実行時はダイアログから名刺画像をアップロードしてください。"
        )

print("名刺画像を処理中...")
with open(image_path,"rb") as file:
    encoded = base64.b64encode(file.read()).decode('utf-8')

ext = os.path.splitext(image_path)[1].lower()[1:]
if ext not in ('png', 'gif', 'bmp', 'webp'):
    ext = 'jpeg'
image_base64 = f"data:image/{ext};base64,{encoded}"

import litellm
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

prompt_text = (
    "提供された名刺画像からテキスト情報を読み取り指定されたフォーマットのみに従って出力してください\n\n"
    "【出力ルール】\n"
    "-挨拶や解説などの余計な文字列は一切出力せず以下のフォーマットのみを出力してください\n"
    "-項目が存在しないまたは読み取れない場合は「なし」と出力してください\n\n"
    "【出力フォーマット】\n"
    "Company_name: 会社名\n"
    "Department: 部署名\n"
    "Title: 役職\n"
    "Name: 氏名\n"
    "Name_kana: フリガナ\n"
    "Email: メールアドレス\n"
    "Phone: 電話番号\n"
    "Address: 住所\n"
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": prompt_text
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": image_base64
                }
            }
        ]
    }
]

response = litellm.completion(
    model="gemini/gemini-3.6-flash",
    messages=messages
)
ocr_result_text = response.choices[0].message.content

card = BusinessCard(ocr_result_text)
print("抽出されたプロファイル情報:")
print(card)



---名刺画像の読み込み---
※ファイルをアップロードするか、キャンセル/スキップして同梱のサンプル画像を使用します。


同梱のサンプル画像を使用します: Gemini_Generated_Image_deywjjdeywjjdeyw (3).jpeg
名刺画像を処理中...
抽出されたプロファイル情報:
会社名:株式会社ソリューション・ブリッジ
部署名:システム開発部
役職:代表取締役
氏名:佐藤 健太
フリガナ:サトウ ケンタ
メールアドレス:k.sato@solubridge.co.jp
電話番号:052-123-4567
住所:〒460-0008 愛知県名古屋市中区栄1-2-3


In [74]:
#テスト用のダミー名刺テキスト
sample_text ="""
Company_name:株式会社テクノロジーラボ
Title:技術開発部 ディレクター
Name:山田 太郎

Address:〒100-0001 東京都千代田区1-1-1
TEL:03-1234-5678
MAIL:yamada@example.com
"""
card = BusinessCard(sample_text)

print("氏名:", card.Name)
print("会社名:", card.Company_name)

氏名: 山田 太郎
会社名: 株式会社テクノロジーラボ


In [75]:
!pip install pandas openpyxl

In [76]:
from openpyxl.chart.layout import Layout, ManualLayout
from openpyxl.chart.axis import ChartLines
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import BarChart, Reference

def export_cards_to_excel(card_data_list, output_filename="business_cards_output.xlsx"):
    """
    名刺データの辞書リストを受け取り、デザインされたExcelファイルを出力する関数
    """
    if not card_data_list:
        print("出力するデータがありません。")
        return

    df = pd.DataFrame(card_data_list)

    column_mapping = {
        "company": "会社名",
        "department": "部署名",
        "title": "役職",
        "name": "名前",
        "name_kana": "フリガナ",
        "email": "メールアドレス",
        "phone": "電話番号",
        "address": "住所"
    }
    df = df.rename(columns=column_mapping)

    wb = Workbook()

    ws_data = wb.active
    ws_data.title = "名刺一覧"

    for r in dataframe_to_rows(df, index=False, header=True):
        ws_data.append(r)

    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Yu Gothic", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Yu Gothic", size=10)
    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'),
        right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'),
        bottom=Side(style='thin', color='D9D9D9')
    )

    for col in ws_data.columns:
        max_len = 0
        col_letter = col[0].column_letter

        for cell in col:
            cell.border = thin_border
            if cell.row == 1:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal="center", vertical="center")
            else:
                cell.font = data_font
                cell.alignment = Alignment(vertical="center")

            val_str = str(cell.value or '')

            str_len = sum(2 if ord(c) > 256 else 1 for c in val_str)
            max_len = max(max_len, str_len)

        ws_data.column_dimensions[col_letter].width = max(max_len + 3, 12)
        ws_summary = wb.create_sheet(title="集計サマリー")

        if "会社名" in df.columns:
            company_counts = df["会社名"].value_counts().reset_index()
            company_counts.columns = ["会社名", "件数"]

            ws_summary.cell(row=1, column=1, value="■ 会社登録件数サマリー").font = Font(name="Yu Gothic", size=12, bold=True)
            start_row = 3
            for r_idx, row in enumerate(dataframe_to_rows(company_counts, index=False, header=True), start=start_row):
                for c_idx, val in enumerate(row, start=1):
                    cell = ws_summary.cell(row=r_idx, column=c_idx, value=val)
                    cell.border = thin_border
                    if r_idx == start_row:
                        cell.fill = header_fill
                        cell.font = header_font
                        cell.alignment = Alignment(horizontal="center")
                    else:
                        cell.font = data_font
            ws_summary.column_dimensions['A'].witch = 35
            ws_summary.column_dimensions['B'].witch = 12

            num_rows = len(company_counts)
            chart = BarChart()
            chart.type = "col"
            chart.title = "会社別名刺件数"
            chart.y_axis.title = "件数"
            chart.x_axis.title = "会社名"

            chart.title.overlay = False

            chart.width = 18
            chart.height = 12
            chart.x_axis.tickLblPos = "nextTo"

            gridlines = ChartLines()
            chart.y_axis.majorGridlines = gridlines

            chart.plot_area.layout = Layout(
                manualLayout=ManualLayout(
                    x=0.1,
                    y=0.18,
                    h=0.72,
                    w=0.8
                )
            )

            data_ref = Reference(ws_summary, min_col=2, min_row=start_row, max_row=start_row + num_rows)
            cats_ref = Reference(ws_summary, min_col=1, min_row=start_row + 1, max_row=start_row + num_rows)

            chart.add_data(data_ref, titles_from_data=True)
            chart.set_categories(cats_ref)
            chart.legend = None

            ws_summary.add_chart(chart, "D3")

    wb.save(output_filename)
    print(f"✅Excelファイルを書き出しました: {output_filename}")

In [77]:
# テスト用データ
sample_extracted_cards = [
    {
        "company": "株式会社テクノロジーソリューションズ",
        "department": "DX推進部",
        "position": "代表取締役社長",
        "name": "山田 太郎",
        "name_kana": "ヤマダ タロウ",
        "email": "t.yamada@example-tech.co.jp",
        "phone": "03-1234-5678",
        "address": "東京都千代田区大手町1-2-3 テクノタワー10F"
    },
    {
        "company": "株式会社テクノロジーソリューションズ",
        "department": "営業本部 第一営業部",
        "position": "部長",
        "name": "鈴木 花子",
        "name_kana": "スズキ ハナコ",
        "email": "h.suzuki@example-tech.co.jp",
        "phone": "03-1234-5679",
        "address": "東京都千代田区大手町1-2-3 テクノタワー10F"
    },
    {
        "company": "イノベーションパートナーズ合同会社",
        "department": "コンサルティング事業部",
        "position": "シニアマネージャー",
        "name": "佐藤 健一",
        "name_kana": "サトウ ケンイチ",
        "email": "k.sato@innovation-p.jp",
        "phone": "06-9876-5432",
        "address": "大阪府大阪市北区梅田2-4-9 イノベートビル5F"
    }
]

export_cards_to_excel(sample_extracted_cards, "business_cards_test.xlsx")

✅Excelファイルを書き出しました: business_cards_test.xlsx
